In [2]:
# To get matches Fixtures
import os
import requests
import pandas as pd

# ==========================================
# ⚙️ CONFIGURATION (Direct API-Sports Setup)
# ==========================================
# ⚠️ Get this key from https://dashboard.api-football.com/ under Account -> My Access
API_KEY = "0b0f46a2fc8885990970cc4d513b62de" 

BASE_URL = "https://v3.football.api-sports.io/fixtures"
EPL_LEAGUE_ID = 39  # 🏆 EPL is always 39
TARGET_SEASON = 2022  # 📅 Select your historical season

HEADERS = {
    "x-apisports-key": API_KEY  # Direct authentication header
}

def fetch_epl_fixtures(season):
    """
    Fetches all finished matches (fixtures) for the EPL in a specific season
    using the direct API-Sports endpoints.
    """
    query_params = {
        "league": EPL_LEAGUE_ID,
        "season": season
    }
    
    try:
        print(f"📡 Connecting to direct API-Sports server...")
        print(f"📡 Fetching EPL fixtures for the {season} season...")
        
        response = requests.get(BASE_URL, headers=HEADERS, params=query_params)
        response.raise_for_status()
        
        data = response.json()
        
        # Checking for API-specific errors in the JSON body
        if data.get("errors"):
            print(f"❌ API Error returned: {data['errors']}")
            return []
            
        fixtures_list = data.get("response", [])
        
        # Filter for Finished Matches (FT) only to ensure complete training data
        finished_fixtures = []
        for match in fixtures_list:
            if match["fixture"]["status"]["short"] in ["FT", "AET", "PEN"]:
                finished_fixtures.append({
                    "fixture_id": match["fixture"]["id"],
                    "date": match["fixture"]["date"],
                    "home_team": match["teams"]["home"]["name"],
                    "away_team": match["teams"]["away"]["name"],
                    "home_goals": match["goals"]["home"],
                    "away_goals": match["goals"]["away"],
                    "status": match["fixture"]["status"]["short"]
                })
                
        print(f"✅ Successfully extracted {len(finished_fixtures)} finished EPL matches.")
        return finished_fixtures
        
    except requests.exceptions.RequestException as e:
        print(f"❌ HTTP Connection failed: {e}")
        return []

# ==========================================
# 🚀 RUNNER
# ==========================================
if __name__ == "__main__":
    fixtures = fetch_epl_fixtures(TARGET_SEASON)
    
    if fixtures:
        df = pd.DataFrame(fixtures)
        print("\n🏆 Sample of EPL Finished Fixtures (Ready for statistics harvesting):")
        print("================================================================")
        print(df.head(10).to_string(index=False))
        
        # Save fixtures catalog to a CSV
        df.to_csv(f"epl_fixtures_catalog_{TARGET_SEASON}.csv", index=False)


📡 Connecting to direct API-Sports server...
📡 Fetching EPL fixtures for the 2022 season...
✅ Successfully extracted 380 finished EPL matches.

🏆 Sample of EPL Finished Fixtures (Ready for statistics harvesting):
 fixture_id                      date         home_team         away_team  home_goals  away_goals status
     867946 2022-08-05T19:00:00+00:00    Crystal Palace           Arsenal           0           2     FT
     867947 2022-08-06T11:30:00+00:00            Fulham         Liverpool           2           2     FT
     867951 2022-08-06T14:00:00+00:00         Newcastle Nottingham Forest           2           0     FT
     867948 2022-08-06T14:00:00+00:00       Bournemouth       Aston Villa           2           0     FT
     867952 2022-08-06T14:00:00+00:00         Tottenham       Southampton           4           1     FT
     867949 2022-08-06T14:00:00+00:00             Leeds            Wolves           2           1     FT
     867953 2022-08-06T16:30:00+00:00           Evert

In [3]:
import os
import time
import requests
import pandas as pd

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
# ⚠️ Paste your direct API key from https://dashboard.api-football.com/
API_KEY = "0b0f46a2fc8885990970cc4d513b62de"

HEADERS = {
    "x-apisports-key": API_KEY
}

# File names
INPUT_CATALOG = "epl_fixtures_catalog_2022.csv"
OUTPUT_FILE = "epl_master_player_stats_2022.csv"

# Safe quota threshold (Stop when you have this many requests remaining for the day)
SAFE_REMAINING_QUOTA = 5 

def fetch_player_stats_with_headers(fixture_id):
    """
    Fetches stats and returns both the player data and the API quota status.
    """
    url = "https://v3.football.api-sports.io/fixtures/players"
    params = {"fixture": fixture_id}
    
    try:
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        
        # Read the API rate-limit headers
        # API-Sports returns 'x-ratelimit-requests-remaining' in headers
        remaining = response.headers.get("x-ratelimit-requests-remaining")
        limit = response.headers.get("x-ratelimit-requests-limit")
        
        remaining_quota = int(remaining) if remaining is not None else 100
        limit_quota = int(limit) if limit is not None else 100
        
        data = response.json()
        
        # Check for any API-level errors
        if data.get("errors") and data.get("errors") != []:
            print(f"\n⚠️ API Error on fixture {fixture_id}: {data['errors']}")
            return None, remaining_quota, limit_quota
            
        return data.get("response", []), remaining_quota, limit_quota
        
    except requests.exceptions.RequestException as e:
        print(f"\n❌ Network error on fixture {fixture_id}: {e}")
        # Return generic default quotas if request fails to prevent script crash
        return None, 100, 100

def parse_player_stats(raw_data, fixture_id):
    """
    Parses raw statistics into tabular player records.
    """
    parsed_records = []
    if not raw_data:
        return parsed_records

    for team_data in raw_data:
        team_name = team_data["team"]["name"]
        for player_entry in team_data["players"]:
            player = player_entry["player"]
            if not player_entry["statistics"]:
                continue
            stats = player_entry["statistics"][0]
            
            record = {
                "fixture_id": fixture_id,
                "team": team_name,
                "player_id": player.get("id"),
                "player_name": player.get("name"),
                "minutes_played": stats["games"].get("minutes"),
                "rating": stats["games"].get("rating"),
                "position": stats["games"].get("position"),
                "is_substitute": stats["games"].get("substitute"),
                "shots_total": stats["shots"].get("total"),
                "shots_on_goal": stats["shots"].get("on"),
                "goals_total": stats["goals"].get("total"),
                "assists": stats["goals"].get("assists"),
                "passes_total": stats["passes"].get("total"),
                "passes_key": stats["passes"].get("key"),
                "passes_accuracy_pct": stats["passes"].get("accuracy"),
                "tackles_total": stats["tackles"].get("total"),
                "tackles_interceptions": stats["tackles"].get("interceptions"),
                "duels_won": stats["duels"].get("won"),
            }
            parsed_records.append(record)
    return parsed_records

# ==========================================
# 🚀 RUNNER
# ==========================================
if __name__ == "__main__":
    if API_KEY == "YOUR_DIRECT_API_SPORTS_KEY_HERE":
        print("🛑 ERROR: Replace 'YOUR_DIRECT_API_SPORTS_KEY_HERE' with your actual key before running.")
        exit()

    # Load Catalog
    if not os.path.exists(INPUT_CATALOG):
        print(f"🛑 ERROR: Could not find catalog file '{INPUT_CATALOG}' in this directory.")
        exit()
        
    fixtures_df = pd.read_csv(INPUT_CATALOG)
    fixture_ids = fixtures_df["fixture_id"].tolist()
    print(f"📊 Loaded {len(fixture_ids)} EPL matches from the catalog.")

    # Check for existing progress so we do not fetch the same match twice
    harvested_fixtures = set()
    if os.path.exists(OUTPUT_FILE):
        existing_df = pd.read_csv(OUTPUT_FILE)
        if "fixture_id" in existing_df.columns:
            harvested_fixtures = set(existing_df["fixture_id"].unique())
            print(f"🔄 Found existing progress. Skipping {len(harvested_fixtures)} already-harvested matches.")

    # Filter out fixtures we have already processed
    fixtures_to_harvest = [fid for fid in fixture_ids if fid not in harvested_fixtures]
    
    if not fixtures_to_harvest:
        print("🎉 All matches in the catalog have already been successfully processed!")
        exit()

    print(f"🏃 Ready to harvest {len(fixtures_to_harvest)} remaining matches.")
    print(f"🔒 Throttling active: waiting 6.5 seconds between matches to maintain <10 requests/minute.")

    for idx, fid in enumerate(fixtures_to_harvest):
        print(f"\n📡 [{idx+1}/{len(fixtures_to_harvest)}] Fetching Fixture ID: {fid}...")
        
        # 1. Fetch match stats and remaining quota
        raw_stats, remaining, limit = fetch_player_stats_with_headers(fid)
        print(f"📉 API Usage: {remaining}/{limit} daily requests remaining.")
        
        # 2. Parse and save progress immediately (Append Mode)
        if raw_stats:
            parsed = parse_player_stats(raw_stats, fid)
            if parsed:
                temp_df = pd.DataFrame(parsed)
                # If file doesn't exist, write headers. If it does, append without headers.
                header_flag = not os.path.exists(OUTPUT_FILE)
                temp_df.to_csv(OUTPUT_FILE, mode='a', index=False, header=header_flag)
                print(f"✅ Successfully appended {len(parsed)} player records to '{OUTPUT_FILE}'.")
        
        # 3. Check quota health. Stop before we completely run out.
        if remaining <= SAFE_REMAINING_QUOTA:
            print(f"\n🛑 QUOTA STOP TRIGGERED: You have reached your safe remaining quota boundary ({remaining}/{limit}).")
            print("To protect your account from suspension or errors, processing has paused.")
            print("Run this script again tomorrow to pick up exactly where you left off!")
            break
            
        # 4. Mandatory sleep time to strictly stay below the 10 requests per minute limit
        # (60 seconds / 10 requests = 6 seconds pause + 0.5 second buffer)
        time.sleep(6.5)

    print("\n🏁 Session closed. All retrieved data is fully secured in your CSV file.")


📊 Loaded 380 EPL matches from the catalog.
🏃 Ready to harvest 380 remaining matches.
🔒 Throttling active: waiting 6.5 seconds between matches to maintain <10 requests/minute.

📡 [1/380] Fetching Fixture ID: 867946...
📉 API Usage: 88/100 daily requests remaining.
✅ Successfully appended 40 player records to 'epl_master_player_stats_2022.csv'.

📡 [2/380] Fetching Fixture ID: 867947...
📉 API Usage: 87/100 daily requests remaining.
✅ Successfully appended 40 player records to 'epl_master_player_stats_2022.csv'.

📡 [3/380] Fetching Fixture ID: 867951...
📉 API Usage: 86/100 daily requests remaining.
✅ Successfully appended 40 player records to 'epl_master_player_stats_2022.csv'.

📡 [4/380] Fetching Fixture ID: 867948...
📉 API Usage: 85/100 daily requests remaining.
✅ Successfully appended 40 player records to 'epl_master_player_stats_2022.csv'.

📡 [5/380] Fetching Fixture ID: 867952...
📉 API Usage: 84/100 daily requests remaining.
✅ Successfully appended 40 player records to 'epl_master_play